In [ ]:
# Import the needed libraries
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

In [ ]:
# Check the TopologicPy Version
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

In [ ]:
# Set my renderer: ( Visual studio code: "vscode" /  Google Colab: "colab" / Browser: "browser" )
renderer = "vscode"

In [ ]:
# Import the OBJ file as a Topology object
objects = Topology.ByOBJPath(r"C:\Users\charl\Desktop\iaac\MaCad\01-year-I\03-module-3\aia26 - s.3 graph ml\GML_macad_26\Graph-ML-Seminar\01-Graph Generation\assets\courthouse_new.obj")
print(objects)

In [ ]:
cells = [] # empty list to store the cells ( rooms )
selectors = [] # Loop through the objects, get their faces, create cells, remove collinear edges, get internal vertices, set colors and vertex sizes based on the name of the object, and store the cells and selectors in the respective lists
for object in objects:
    d = Topology.Dictionary(object) # get the dictionary of the object to access its name and set new values for color and vertex size
    faces = Topology.Faces(object)  # get the faces of the object to create a cell from them, and to check if the object has more than one face to be considered as a room
    if len(faces) > 1: 
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c) # remove collinear edges to avoid having multiple vertices at the same location
        s = Topology.InternalVertex(c) # get the internal vertices of the cell to use them as selectors for the rooms
        name = Dictionary.ValueAtKey(d, "name") # get the name of the object to set the color of the cell based on it
        if "public" in name:
            color = "yellow"
        if "jury" in name:
            color = "red"
        if "witness" in name:
            color = "orange"
        if "judge" in name:
            color = "blue"
        if "prisoner" in name:
            color = "black"
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 15]) # set the color and vertex size in the dictionary to be used by the renderer
        s = Topology.SetDictionary(s, d) # set the dictionary of the selector to have the same color and vertex size as the cell
        selectors.append(s) # store the selector in the list to render them later
        cells.append(c) # store the cell in the list to render them later
        print(Dictionary.Keys(d), Dictionary.Values(d)) # print the keys and values of the dictionary to check if the color and vertex size are set correctly

print(len(cells))

In [ ]:
house = CellComplex.ByCells(cells) # create a cell complex from the cells
house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells=True) # transfer the dictionaries from the selectors to the cells
house_cells = Topology.Cells(house) # get the cells of the cell complex to check if the dictionaries are transferred correctly, and to print their keys and values
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

In [ ]:
# Show the house
Topology.Show(house_cells, 
              faceColorKey="color", 
              faceOpacity=1.0,
              opacityKey="nothing",
              faceOpacityKey="nothing",
              backgroundColor="white",  
              width=700,
              height=500,
              edgeColor="white")

In [ ]:
# Create a primal graph
g = Graph.ByTopology(house) 
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

In [ ]:
# Show the graph and the house
Topology.Show(g, house,
              vertexSizeKey="vertex_size",
              vertexColorKey="color",
              backgroundColor="white",  
              width=700,
              height=500)


In [ ]:
# Import the doors from the OBJ file, remove collinear edges, set their type, color, and vertex size in their dictionary based on the object
apertures = []
objects = Topology.ByOBJPath(r"C:\Users\charl\Desktop\iaac\MaCad\01-year-I\03-module-3\aia26 - s.3 graph ml\GML_macad_26\Graph-ML-Seminar\01-Graph Generation\assets\courthouse_doors_new.obj")
for object in objects:
    d = Topology.Dictionary(object)  # get the dictionary to read the name
    name = Dictionary.ValueAtKey(d, "name") or ""
    
    if "public" in name:
        color = "yellow"
    elif "jury" in name:
        color = "red"
    elif "judge" in name:
        color = "blue"
    elif "prisoner" in name:
        color = "black"
    elif "witness" in name:
        color = "orange"
    else:
        print(f"⚠️ Skipping object with unrecognized name: '{name}'")
        continue
    
    faces = Topology.Faces(object)
    if not faces:
        print(f"⚠️ No faces found for object: '{name}' (type: {Topology.Type(object)})")
        continue
    
    face = faces[0]
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["door", color, 12])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)

print(apertures)
print(len(apertures))

In [ ]:
# Import the windows from the OBJ file, remove collinear edges, set their type, color, and vertex size in their dictionary, and store them in apertures
objects = Topology.ByOBJPath(r"C:\Users\charl\Desktop\iaac\MaCad\01-year-I\03-module-3\aia26 - s.3 graph ml\GML_macad_26\Graph-ML-Seminar\01-Graph Generation\assets\courthouse_windows_new.obj")

if objects is None:
    raise FileNotFoundError("OBJ file not found or could not be loaded. Check the path.")

for object in objects:
    d = Topology.Dictionary(object)
    name = Dictionary.ValueAtKey(d, "name") or ""
    
    faces = Topology.Faces(object)
    if not faces:
        print(f"⚠️ No faces found for object: '{name}' (type: {Topology.Type(object)})")
        continue
    
    face = faces[0]
    face = Topology.RemoveCollinearEdges(face)
    d = Dictionary.ByKeysValues(["type", "color", "vertex_size"], ["window", "cyan", 8])
    face = Topology.SetDictionary(face, d)
    apertures.append(face)

print(apertures)
print(len(apertures))

In [ ]:
house = Topology.AddApertures(house, apertures, subTopologyType="face")

In [ ]:
# Create a dual graph 
g = Graph.ByTopology(house, direct=False, viaSharedApertures=True, toExteriorApertures=True)
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

In [ ]:
Topology.Show(house, apertures, g, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="white", width=1080, height=600)